## Silver tables load with DQ checks, SCD type 1 and Type 2 tables (Apply changes API )

In [0]:
import dlt
from pyspark.sql.functions import expr, regexp_extract, col, current_timestamp

In [0]:
# dq checks - Use this variable in dlt.expect for separating valid and invalid records

checks = {}

checks["valid_customer"] = "(customer_id IS NOT NULL)"
checks["valid_customer_name"] = "(trim(first_name) <> '' or first_name IS NOT NULL)"

dq_checks = "({0})".format(" AND ".join(checks.values()))


##### here i am creating stream tables even for staging but we can create views if we want, usually staging will always be truncate and insert records so that main table will always be a merge

In [0]:
# read bronze streaming table - CDC is enabled

# customers silver table with filtered good records

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    },
    comment="Staging Customer Silver Table",
    name="stg_customers"
)
@dlt.expect_all(checks)
def customers():
    df_customers = (
        spark.readStream.table("deltalake.new_bronze.customers")
    )

    df_customers_new = df_customers.withColumns({
        "full_name": expr("concat(first_name,' ',last_name)"),
        "email_domain": regexp_extract(col("email"),'@([^\\.]+)',1),
        "last_updated_timestamp": current_timestamp(),
        "dq_check": expr(dq_checks)
    })

    df_filtered = df_customers_new.filter("dq_check=true")
    
    return df_filtered

#### SCD - 1, keys - customerid , seq by registration_date

In [0]:
# read from stage customers post transformation and create SCD 1 logic for customers table.

# SCD - 1, keys - customerid , seq by registration_date

dlt.create_streaming_table(
    name="customers",
    comment="Customer Silver Table",
    table_properties={
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    }
)

dlt.apply_changes(
        target="customers",
        source="stg_customers",
        keys=["customer_id"],
        sequence_by = col("registration_date"),
        stored_as_scd_type=1
    )

# updating all the columns if needed specify columns List or except columns

In [0]:
# customers silver table with error records

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    },
    comment="Staging Customer Silver Error Table",
    name="stg_customers_error"
)

@dlt.expect_all(checks)
def customers():
    df_customers = (
        spark.readStream.table("deltalake.new_bronze.customers")
    )
    df_customers_new = df_customers.withColumns({
        "dq_check": expr(dq_checks)
    })

    df_filtered = df_customers_new.filter("dq_check=false")
    
    return df_filtered

### similar for products - dq checks can be implemented, i am only implementing only SCD Type - 2

In [0]:
# Products silver table

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false",
        "delta.feature.timestampNtz": "supported"
    },
    comment="Staging Products Silver Table",
    name="stg_products"
)
def products():
    df=(
        spark.readStream.table("deltalake.new_bronze.products")
    )

    df_new = df.withColumn("last_updated_timestamp", current_timestamp())

    return df_new

In [0]:
# scd type 2 for products table 

# read from stg_silver_products and create SCD 2 logic for products table maintaining history

dlt.create_streaming_table(
    name="products",
    comment="Silver Products Table",
    table_properties={
        "quality": "silver",
        "pipelines.reset.allowed": "false",
        "delta.feature.timestampNtz": "supported"
    }
)

dlt.apply_changes(
        target="products",
        source="stg_products",
        keys=["product_id"],
        sequence_by = col("launch_date"),
        apply_as_deletes=expr("is_active = 'false'"),
        stored_as_scd_type=2
    )

# updating all the columns if needed specify columnlist or except columns

In [0]:
# Orders silver table

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    },
    comment="Orders Silver Table",
    name="orders"
)
def customers():
    df=(
        spark.readStream.table("deltalake.new_bronze.orders")
    )
    
    df_new = df.withColumn("last_updated_timestamp", current_timestamp())

    return df_new

In [0]:
#joined silver table 

@dlt.table(
    name = "fact_transaction",
    table_properties = {
        "quality": "silver",
        "delta.feature.timestampNtz": "supported"
    },
    comment="Silver Final Transaction Table",
)
def fact_policy_transaction():

    df_customers = spark.readStream.table("LIVE.customers")
    df_products = spark.readStream.table("LIVE.products")
    df_orders = spark.readStream.table("LIVE.orders")

    df_joined =(
        df_orders
        .join(df_customers, df_orders.customer_id == df_customers.customer_id, "inner")
        .join(df_products, df_orders.product_id == df_products.product_id, "inner")
        .select(
            df_customers.customer_id.alias("customer_id"),
            df_customers.full_name,
            df_customers.email,
            df_customers.date_of_birth,
            df_customers.email_domain,
            df_customers.address,
            df_customers.phone_number,
            df_customers.registration_date,
            df_products.product_id.alias("product_id"),
            df_products.product_name,
            df_products.product_type,
            df_products.coverage_details,
            df_products.premium_base_rate,
            df_products.launch_date,
            df_products.is_active,
            df_products.__START_AT.alias("start_at"),
            df_products.__END_AT.alias("end_at"),
            df_orders.policy_id.alias("policy_id"),
            df_orders.policy_start_date,
            df_orders.policy_end_date,
            df_orders.premium_amount,
            df_orders.payment_frequency,
            df_orders.policy_status,
            df_orders.last_payment_date,
            df_orders.renewal_date
            )
    )

    return df_joined